# Dataset Prep

This notebook splits labeled orthophotos from an input directory into `train`, `val`, and `test` dataset folders.

In [3]:
import importlib
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils import data_utils

importlib.reload(data_utils)

<module 'utils.data_utils' from '/lustre06/project/6007330/keyuyao/power-grid-detection/src/utils/data_utils.py'>

## Configuration

Edit these paths before running the notebook.

This notebook splits photos directly from `LABELED_PHOTOS_DIR` into `train/`, `val/`, and `test/` folders, each with an `images/` directory and an `annotations.coco.json` file.

In [4]:
DATA_ROOT = Path("/home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data")
LABELED_PHOTOS_DIR = DATA_ROOT / "labeled"
SPLITS_DIR = DATA_ROOT / "splits"
TILINGS_DIR = DATA_ROOT / "tilings"
LABELS_PATH = DATA_ROOT / "annotations" / "bounding_box.gpkg"

# Update this to match the class/category column in your GeoPackage.
CLASS_FIELD = "class"
TILE_SIZE = 512
STRIDE = 256
MIN_AREA_PX = 1.0

RATIOS = (0.7, 0.15, 0.15)
SEED = 42

print(f"Labeled photos dir: {LABELED_PHOTOS_DIR}")
print(f"Splits dir: {SPLITS_DIR}")
print(f"Tilings dir: {TILINGS_DIR}")
print(f"Labels path: {LABELS_PATH}")

Labeled photos dir: /home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/labeled
Splits dir: /home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/splits
Tilings dir: /home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/tilings
Labels path: /home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/annotations/bounding_box.gpkg


In [5]:
splits = data_utils.split_photo_dir(
    photos_dir=LABELED_PHOTOS_DIR,
    output_dir=SPLITS_DIR,
    ratios=RATIOS,
    seed=SEED,
)

for split_name, split_df in splits.items():
    split_dir = SPLITS_DIR / split_name
    images_dir = split_dir / "images"
    annotations_path = split_dir / "annotations.coco.json"
    print(f"{split_name}: {len(split_df)} photos")
    print(f"  images dir: {images_dir}")
    print(f"  annotations: {annotations_path}")
    display(split_df.head())

print(f"Saved split dataset folders to {SPLITS_DIR}")

train: 8 photos
  images dir: /home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/splits/train/images
  annotations: /home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/splits/train/annotations.coco.json


,filename,source_path,split
0,geomont_orthophoto2020_20cm_2950_230_5030.tif,/home/keyuyao/projects/def-sdaniel/keyuyao/pow...,train
1,geomont_orthophoto2020_20cm_2950_235_4985.tif,/home/keyuyao/projects/def-sdaniel/keyuyao/pow...,train
2,geomont_orthophoto2020_20cm_2950_250_5010.tif,/home/keyuyao/projects/def-sdaniel/keyuyao/pow...,train
3,geomont_orthophoto2020_20cm_2950_345_5005.tif,/home/keyuyao/projects/def-sdaniel/keyuyao/pow...,train
4,geomont_orthophoto2020_20cm_2950_345_5090.tif,/home/keyuyao/projects/def-sdaniel/keyuyao/pow...,train


val: 2 photos
  images dir: /home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/splits/val/images
  annotations: /home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/splits/val/annotations.coco.json


,filename,source_path,split
0,geomont_orthophoto2020_20cm_2950_270_5015.tif,/home/keyuyao/projects/def-sdaniel/keyuyao/pow...,val
1,geomont_orthophoto2020_20cm_2950_360_5055.tif,/home/keyuyao/projects/def-sdaniel/keyuyao/pow...,val


test: 1 photos
  images dir: /home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/splits/test/images
  annotations: /home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/splits/test/annotations.coco.json


,filename,source_path,split
0,geomont_orthophoto2020_20cm_2950_350_5030.tif,/home/keyuyao/projects/def-sdaniel/keyuyao/pow...,test


Saved split dataset folders to /home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/splits


## Tiling
Tile both photos and labels

In [7]:
if not LABELS_PATH.exists():
    raise FileNotFoundError(
        f"Labels file not found: {LABELS_PATH}. Put your GeoPackage there or update LABELS_PATH."
    )

import json
import random
import shutil

import geopandas as gpd
import rasterio
from rasterio.windows import Window
from rasterio.windows import bounds as window_bounds
from shapely.geometry import box

from utils.proj_utils import configure_proj_data_dir

PAD = False
KEEP_EMPTY_TILES = False
EMPTY_TILE_FRACTION = 0.01
MIN_BOX_AREA_PX = int(MIN_AREA_PX)
MIN_BOX_SIDE_PX = 1
DROP_BLACK_TILES = False
BLACK_THRESHOLD = 0.98

configure_proj_data_dir()

def iter_tile_origins(width: int, height: int, stride: int) -> list[tuple[int, int]]:
    return [(x, y) for y in range(0, height, stride) for x in range(0, width, stride)]

def build_tile_window(src_width: int, src_height: int, x: int, y: int, tile_size: int, pad: bool) -> tuple[Window, int, int]:
    if pad:
        return Window(col_off=x, row_off=y, width=tile_size, height=tile_size), tile_size, tile_size
    tile_w = min(tile_size, src_width - x)
    tile_h = min(tile_size, src_height - y)
    return Window(col_off=x, row_off=y, width=tile_w, height=tile_h), int(tile_w), int(tile_h)

def bbox_world_to_pixel(transform, width: int, height: int, xmin: float, ymin: float, xmax: float, ymax: float) -> tuple[float, float, float, float]:
    inv = ~transform
    left, top = inv * (xmin, ymax)
    right, bottom = inv * (xmax, ymin)
    x0 = max(0.0, min(float(width), min(left, right)))
    y0 = max(0.0, min(float(height), min(top, bottom)))
    x1 = max(0.0, min(float(width), max(left, right)))
    y1 = max(0.0, min(float(height), max(top, bottom)))
    return x0, y0, x1 - x0, y1 - y0

def is_black_tile(tile, black_threshold: float = 0.98) -> bool:
    if tile.size == 0:
        return True
    return float((tile <= 1).mean()) >= black_threshold

if not LABELS_PATH.exists():
    raise FileNotFoundError(
        f"Labels file not found: {LABELS_PATH}. Put your GeoPackage there or update LABELS_PATH."
    )

labels_gdf = gpd.read_file(LABELS_PATH)
if labels_gdf.empty:
    raise ValueError(f"No geometries found in {LABELS_PATH}")
if CLASS_FIELD not in labels_gdf.columns:
    raise ValueError(f"Class column {CLASS_FIELD!r} not found. Available columns: {list(labels_gdf.columns)}")
if labels_gdf.crs is None:
    raise ValueError("Input geopackage has no CRS; cannot align annotations with rasters.")

labels_gdf = labels_gdf.copy()
labels_gdf[CLASS_FIELD] = labels_gdf[CLASS_FIELD].astype(str).str.strip()
class_names = sorted(labels_gdf[CLASS_FIELD].dropna().unique().tolist())
class_to_id = {name: idx + 1 for idx, name in enumerate(class_names)}
categories = [{"id": cid, "name": name, "supercategory": "object"} for name, cid in class_to_id.items()]

print(f"Loaded {len(labels_gdf)} features from {LABELS_PATH}")
print(f"Classes: {class_to_id}")

tiling_summaries = {}
rng = random.Random(SEED)

for split_name in ("train", "val", "test"):
    split_images_dir = SPLITS_DIR / split_name / "images"
    split_output_dir = TILINGS_DIR / split_name
    split_tiles_dir = split_output_dir / "images"
    annotations_path = split_output_dir / "annotations.coco.json"

    if split_output_dir.exists():
        shutil.rmtree(split_output_dir)
    split_tiles_dir.mkdir(parents=True, exist_ok=True)

    images = []
    annotations = []
    image_id = 1
    annotation_id = 1
    tiles_skipped_empty = 0
    tiles_skipped_black = 0
    source_images = sorted(
        p for p in split_images_dir.iterdir() if p.is_file() and p.suffix.lower() in {".tif", ".tiff"}
    )

    for src_path in source_images:
        with rasterio.open(src_path) as src:
            labels = labels_gdf if str(labels_gdf.crs) == str(src.crs) else labels_gdf.to_crs(src.crs)
            src_poly = box(*src.bounds)
            labels_in_src = labels[labels.geometry.intersects(src_poly)]
            profile = src.profile.copy()

            for x, y in iter_tile_origins(src.width, src.height, STRIDE):
                window, tile_w, tile_h = build_tile_window(src.width, src.height, x, y, TILE_SIZE, PAD)
                if tile_w <= 0 or tile_h <= 0:
                    continue

                if PAD:
                    tile = src.read(window=window, boundless=True, fill_value=0)
                else:
                    tile = src.read(window=window)

                if DROP_BLACK_TILES and is_black_tile(tile, black_threshold=BLACK_THRESHOLD):
                    tiles_skipped_black += 1
                    continue

                tile_transform = src.window_transform(window)
                tile_poly = box(*window_bounds(window, src.transform))
                labels_in_tile = labels_in_src[labels_in_src.geometry.intersects(tile_poly)]
                tile_annotations = []

                if not labels_in_tile.empty:
                    for _, row in labels_in_tile.iterrows():
                        geom = row.geometry
                        if geom is None or geom.is_empty:
                            continue

                        clipped = geom.intersection(tile_poly)
                        if clipped.is_empty:
                            continue

                        xmin, ymin, xmax, ymax = clipped.bounds
                        if xmax <= xmin or ymax <= ymin:
                            continue

                        bx, by, bw, bh = bbox_world_to_pixel(tile_transform, tile_w, tile_h, xmin, ymin, xmax, ymax)
                        area = float(bw * bh)
                        if bw < MIN_BOX_SIDE_PX or bh < MIN_BOX_SIDE_PX or area < MIN_BOX_AREA_PX:
                            continue

                        category_id = class_to_id[str(row[CLASS_FIELD])]
                        tile_annotations.append(
                            {
                                "id": annotation_id,
                                "image_id": image_id,
                                "category_id": category_id,
                                "bbox": [bx, by, bw, bh],
                                "area": area,
                                "iscrowd": 0,
                                "segmentation": [[bx, by, bx + bw, by, bx + bw, by + bh, bx, by + bh]],
                            }
                        )
                        annotation_id += 1

                has_annotations = len(tile_annotations) > 0
                keep_tile = has_annotations or (KEEP_EMPTY_TILES and rng.random() < EMPTY_TILE_FRACTION)
                if not keep_tile:
                    tiles_skipped_empty += 1
                    continue

                tile_name = f"{src_path.stem}_x{x:05d}_y{y:05d}.tif"
                tile_path = split_tiles_dir / tile_name
                tile_profile = profile.copy()
                tile_profile.update(width=tile_w, height=tile_h, transform=tile_transform)
                if str(tile_profile.get("photometric", "")).upper() == "YCBCR":
                    tile_profile.pop("photometric", None)
                    tile_profile["compress"] = "lzw"

                with rasterio.open(tile_path, "w", **tile_profile) as dst:
                    dst.write(tile)

                images.append(
                    {
                        "id": image_id,
                        "file_name": f"images/{tile_name}",
                        "width": tile_w,
                        "height": tile_h,
                        "source_path": str(tile_path.resolve()),
                        "split": split_name,
                    }
                )
                annotations.extend(tile_annotations)
                image_id += 1

    coco = {
        "info": {"description": f"{split_name} tiled split generated from {LABELS_PATH.name}"},
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": categories,
    }
    annotations_path.write_text(json.dumps(coco, indent=2))

    tiling_summaries[split_name] = {
        "images": len(coco["images"]),
        "annotations": len(coco["annotations"]),
        "tiles_written": len(coco["images"]),
        "tiles_skipped_empty": tiles_skipped_empty,
        "tiles_skipped_black": tiles_skipped_black,
        "output_dir": split_output_dir,
        "annotations_path": annotations_path,
    }

for split_name, summary in tiling_summaries.items():
    print(
        f"{split_name}: {summary['images']} tiles, {summary['annotations']} annotations, "
        f"skipped_empty={summary['tiles_skipped_empty']}, "
        f"skipped_black={summary['tiles_skipped_black']}"
    )
    print(f"  tiles dir: {summary['output_dir'] / 'images'}")
    print(f"  annotations: {summary['annotations_path']}")

tiling_summaries


/home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/.venv/lib/python3.11/site-packages/pyproj/__init__.py:96: UserWarning: Valid PROJ data directory not found. Either set the path using the environmental variable PROJ_DATA (PROJ 9.1+) | PROJ_LIB (PROJ<9.1) or with `pyproj.datadir.set_data_dir`.
  warnings.warn(str(err))


Loaded 239 features from /home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/annotations/bounding_box.gpkg
Classes: {'substation': 1, 'tower': 2, 'wind_turbine': 3}
train: 299 tiles, 313 annotations, skipped_empty=76533, skipped_black=0
  tiles dir: /home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/tilings/train/images
  annotations: /home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/tilings/train/annotations.coco.json
val: 728 tiles, 1078 annotations, skipped_empty=18480, skipped_black=0
  tiles dir: /home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/tilings/val/images
  annotations: /home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/tilings/val/annotations.coco.json
test: 0 tiles, 0 annotations, skipped_empty=9604, skipped_black=0
  tiles dir: /home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/tilings/test/images
  annotations: /home/keyuyao/projects/def-sdaniel/keyuyao/power-g

{'train': {'images': 299,
  'annotations': 313,
  'tiles_written': 299,
  'tiles_skipped_empty': 76533,
  'tiles_skipped_black': 0,
  'output_dir': PosixPath('/home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/tilings/train'),
  'annotations_path': PosixPath('/home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/tilings/train/annotations.coco.json')},
 'val': {'images': 728,
  'annotations': 1078,
  'tiles_written': 728,
  'tiles_skipped_empty': 18480,
  'tiles_skipped_black': 0,
  'output_dir': PosixPath('/home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/tilings/val'),
  'annotations_path': PosixPath('/home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/tilings/val/annotations.coco.json')},
 'test': {'images': 0,
  'annotations': 0,
  'tiles_written': 0,
  'tiles_skipped_empty': 9604,
  'tiles_skipped_black': 0,
  'output_dir': PosixPath('/home/keyuyao/projects/def-sdaniel/keyuyao/power-grid-detection/data/tilings/tes